**SAPIANS** · SCC5819 · MÓDULO 00 — A BASE SRAG · A SELEÇÃO, POR DENTRO

# Módulo 00 — internals da seleção: as buscas completas e os braços de medição

O [walkthrough](srag_selection_walkthrough.ipynb) reproduz as partes
baratas do estudo e confere cada número contra o registro. Este caderno
paga o resto da conta: as **buscas inteiras** como saíram
(`gold/selection_metrics.json`), a reprodução de cada topo de busca, e os
braços de medição que sustentam decisões declaradas sem prova no card —
one-hot contra ordinal nas árvores, o preço em AUC da determinância de
semente, a maldição do vencedor, o que custaria retreinar em treino+val.

Duas regras valem aqui e não valem no walkthrough. A primeira: o teste
pode ser lido mais de uma vez, desde que **cada leitura extra seja
rotulada exploratória** (item 6 do critério pré-registrado) — elas não
mudaram a escolha, que já estava feita e registrada. A segunda: a seção
da base cheia depende do Ouro em disco
(`~/Documents/srag-data/gold/`); onde ele não existe, a seção **imprime o
pulo declarado** em vez de fingir, e os outputs commitados vêm de uma
execução local completa.

[README do módulo](../README.md) · [a seleção do modelo](srag_selection_walkthrough.ipynb) · [`SELECTION.md`](../SELECTION.md) · [`MODEL.md`](../MODEL.md) · [`GOLD.md`](../GOLD.md)

---

In [1]:
%pip install -q pandas pyarrow scikit-learn xgboost matplotlib
# versões fixadas em requirements.txt; outras versões movem a terceira casa

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

REPO = pathlib.Path.cwd()
while REPO != REPO.parent and not (REPO / "tools" / "README.md").exists():
    REPO = REPO.parent
if not (REPO / "tools" / "README.md").exists():
    import tempfile
    REPO = pathlib.Path(tempfile.gettempdir()) / "interpretable-ml-lectures"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/wbendinelli/interpretable-ml-lectures.git",
                        str(REPO)], check=True)
sys.path.insert(0, str(REPO / "tools"))
import sapians as SP
import srag_60_model as M
import srag_50_selection as S

RANDOM_STATE = 42
TOL = 1e-3  # a deriva de terceira casa que o ci.yml declara entre plataformas

g = M.load_gold(REPO / "modules/00-dataset/gold/gold_covid_obito_sample.parquet")
SM = json.loads(
    (REPO / "modules/00-dataset/gold/selection_metrics.json").read_text(encoding="utf-8")
)
y = g["y_obito"].to_numpy()
tr = (g["split"] == "train").to_numpy()
va = (g["split"] == "val").to_numpy()
te = (g["split"] == "test").to_numpy()
y_va, y_te = y[va], y[te]
PLACAR = {(r["modelo"], r["split"]): r for r in SM["placar"]}


def compacta(cfg_json):
    """A configuração em uma linha curta, para caber na tabela."""
    return " ".join(f"{k}={v}" for k, v in sorted(json.loads(cfg_json).items()))


def auc_val(nome, cfg):
    """Reajusta no treino e mede na validação — a conta que a busca fez."""
    modelo, segundos = S.fit_one(nome, g, cfg)
    return modelo, round(segundos, 1), S.score(y_va, S.predict(nome, modelo, g, va))["auc"]


OURO_PATH = pathlib.Path.home() / "Documents/srag-data/gold/gold_covid_obito.parquet"
ouro = None
if OURO_PATH.exists():
    ouro = pd.read_parquet(OURO_PATH)
    for c in M.CATEGORICAS:
        ouro[c] = ouro[c].astype("category")
    print(f"Ouro: {SP.pt_int(len(ouro))} x {len(ouro.columns)}")
else:
    print("SEM Ouro local — a seção da base cheia imprime o pulo declarado;")
    print("os outputs commitados vêm de uma execução local completa.")
print(f"amostra: {SP.pt_int(len(g))} linhas;"
      f" val {SP.pt_int(int(va.sum()))}, teste {SP.pt_int(int(te.sum()))}")

Ouro: 1.282.970 x 50
amostra: 240.290 linhas; val 24.148, teste 16.142


## §1 — As buscas, como saíram

Seis buscas, cada uma no seu espaço e no seu orçamento declarado. O que
interessa em cada tabela não é só o topo: é a **amplitude** — de quanto o
modelo se move quando só os hiperparâmetros mudam. É ela que responde
"valeu a pena tunar?" por modelo, e a resposta é diferente para cada
forma funcional.

In [3]:
CASAS_BUSCA = {"auc": 4, "brier": 4, "ece": 4, "fit_s": 1}
for nome in S.MODELOS:
    linhas = pd.DataFrame(SM["buscas"][nome])
    linhas["config"] = linhas["config"].map(compacta)
    tabela = linhas[["posto", "config", "auc", "brier", "ece", "fit_s"]]
    amplitude = round(float(linhas["auc"].max() - linhas["auc"].min()), 4)
    grade = 1
    for valores in SM["espacos"][nome].values():
        grade *= len(valores)
    print(f"=== {nome}: {len(linhas)} de {grade} configurações,"
          f" amplitude de AUC {SP.pt(amplitude, 4)}")
    if len(tabela) <= 8:
        print(SP.tabela(tabela, CASAS_BUSCA))
    else:
        print(f"(as {len(tabela) - 7} configurações do meio ficam de fora da tela)")
        print(SP.tabela(pd.concat([tabela.head(4), tabela.tail(3)]), CASAS_BUSCA))
    print()

=== dummy: 1 de 1 configurações, amplitude de AUC 0,0000
posto         config    auc  brier    ece fit_s
    1 strategy=prior 0,5000 0,1689 0,1243   0,0

=== lpm: 1 de 1 configurações, amplitude de AUC 0,0000
posto config    auc  brier    ece fit_s
    1        0,7232 0,1552 0,1050   1,0

=== logit: 7 de 7 configurações, amplitude de AUC 0,0008
posto  config    auc  brier    ece fit_s
    1 C=0.003 0,7214 0,1545 0,0868   0,7
    2  C=0.01 0,7208 0,1550 0,0877   0,7
    3  C=0.03 0,7206 0,1551 0,0880   0,8
    3   C=0.1 0,7206 0,1552 0,0882   0,8
    3   C=0.3 0,7206 0,1551 0,0879   0,8
    3   C=1.0 0,7206 0,1550 0,0878   0,7
    3   C=3.0 0,7206 0,1550 0,0878   0,8

=== arvore: 24 de 320 configurações, amplitude de AUC 0,0668
(as 17 configurações do meio ficam de fora da tela)
posto                                                               config    auc  brier    ece fit_s
    1 ccp_alpha=0.0 criterion=log_loss max_depth=None min_samples_leaf=200 0,7041 0,1434 0,0354   1,3
    2  

A tunagem vale coisas muito diferentes conforme a forma. A amplitude da
logística cabe folgada dentro de um erro-padrão do bootstrap (§5 do
walkthrough): mexer no `C` aqui é rearranjar ruído. As da árvore e do
XGBoost são dezenas de vezes maiores — e a maior de todas é a da árvore,
que é também o modelo mais simples do zoo depois do LPM.

Isso importa para a honestidade da comparação: um estudo que tunasse só o
modelo favorito e deixasse a logística no default estaria comparando um
candidato com um espantalho. Aqui os seis recebem orçamento declarado, e
o do LPM é 1 porque ele não tem o que tunar.

## §2 — Reprodução: a busca confere?

Uma tabela no JSON é uma alegação. A célula abaixo reajusta cinco
configurações — o topo de `logit`, `arvore`, `xgb` e `floresta`, mais uma
do meio da tabela do XGBoost — e compara a AUC de validação medida agora
com a registrada. É a mesma pilha, a mesma amostra e as mesmas sementes:
o que se testa aqui é se o registro é o que o código produz.

In [4]:
xgb_rows = SM["buscas"]["xgb"]
meio = xgb_rows[len(xgb_rows) // 2]
alvos = [("logit", SM["buscas"]["logit"][0]),
         ("arvore", SM["buscas"]["arvore"][0]),
         ("xgb", xgb_rows[0]),
         ("xgb", meio),
         ("floresta", SM["buscas"]["floresta"][0])]

reproducao = []
for nome, row in alvos:
    cfg = json.loads(row["config"])
    _, segundos, auc = auc_val(nome, cfg)
    reproducao.append({"modelo": nome, "posto": row["posto"],
                       "config": compacta(row["config"]),
                       "auc_registro": row["auc"], "auc_agora": auc,
                       "delta": round(auc - row["auc"], 4), "fit (s)": segundos})
rep = pd.DataFrame(reproducao)
print(SP.tabela(rep, {"auc_registro": 4, "auc_agora": 4, "delta": 4, "fit (s)": 1}))
print(f"\nidênticos em 4 casas: {int((rep['delta'] == 0).sum())}/{len(rep)};"
      f" |delta| máximo {SP.pt(rep['delta'].abs().max(), 4)}")
assert (rep["delta"].abs() <= TOL).all(), "a busca não reproduz o registro"

  modelo posto                                                                              config auc_registro auc_agora  delta fit (s)
   logit     1                                                                             C=0.003       0,7214    0,7214 0,0000     0,8
  arvore     1                ccp_alpha=0.0 criterion=log_loss max_depth=None min_samples_leaf=200       0,7041    0,7041 0,0000     1,6
     xgb     1 learning_rate=0.05 max_depth=4 min_child_weight=100 n_estimators=800 reg_lambda=5.0       0,7564    0,7564 0,0000     4,5
     xgb    16   learning_rate=0.15 max_depth=6 min_child_weight=5 n_estimators=200 reg_lambda=5.0       0,7513    0,7513 0,0000     1,3
floresta     1                                   max_depth=16 max_features=sqrt min_samples_leaf=5       0,7403    0,7403 0,0000    15,5

idênticos em 4 casas: 5/5; |delta| máximo 0,0000


In [5]:
# O atalho de prefixo do `search`: configurações que só diferem em
# n_estimators compartilham UM fit, cortado por iteration_range. Quantos
# grupos a amostra de 30 configurações formou?
grupos = {}
for r in xgb_rows:
    cfg = json.loads(r["config"])
    chave = json.dumps({k: v for k, v in cfg.items() if k != "n_estimators"},
                       sort_keys=True)
    grupos.setdefault(chave, []).append(cfg["n_estimators"])
tamanhos = pd.Series([len(v) for v in grupos.values()]).value_counts().sort_index()
print(f"{len(grupos)} grupos para {len(xgb_rows)} configurações")
print("configurações por grupo:", {int(k): int(v) for k, v in tamanhos.items()})
print("\no ParameterSampler não sorteou duas configurações que só diferissem em")
print("n_estimators: o atalho estava correto e não teve o que economizar aqui.")

30 grupos para 30 configurações
configurações por grupo: {1: 30}

o ParameterSampler não sorteou duas configurações que só diferissem em
n_estimators: o atalho estava correto e não teve o que economizar aqui.


## §3 — Codificação: one-hot ou ordinal para árvore e floresta?

O `tree_matrix` do módulo entrega one-hot completo (`drop_first=False`)
porque a regra `dispneia_sim <= 0.5` é uma frase que se lê em voz alta —
a legibilidade da regra é o produto da árvore, não um detalhe estético. A
alternativa barata seria o **ordinal**: cada categórica vira o código
inteiro do seu nível, e a matriz cai para as 40 colunas das features
originais.

O custo dessa escolha não é presumido zero. A célula abaixo reajusta as
duas configurações vencedoras sobre a matriz ordinal e imprime o delta.
Vale lembrar o que o ordinal faz com um corte: `cs_raca <= 2.5` é uma
regra sobre a ORDEM em que os níveis foram catalogados, que não significa
nada clinicamente — mesmo que a AUC não caia.

In [6]:
def matriz_ordinal(base):
    """40 colunas: numéricas cruas, booleanas 0/1, categóricas em cat.codes."""
    x = base[list(M.NUMERICAS)].copy()
    for c in M.BOOLEANAS:
        x[c] = base[c].astype(int)
    for c in M.CATEGORICAS:
        x[c] = base[c].cat.codes.astype(np.int16)  # NA viraria -1
    return x


x_ord = matriz_ordinal(g)
print(f"matriz ordinal: {x_ord.shape[1]} colunas"
      f" (one-hot: {S.matrices(g)['arvore'].shape[1]})")
print("códigos -1 (nível ausente) entre as categóricas:",
      int((x_ord[list(M.CATEGORICAS)] < 0).sum().sum()))

linhas = []
for nome in ("arvore", "floresta"):
    modelo = S.build(nome, S.MELHORES[nome])
    modelo.fit(x_ord[tr], y[tr])
    auc_ord = S.score(y_va, modelo.predict_proba(x_ord[va])[:, 1])["auc"]
    onehot = PLACAR[(nome, "val")]
    linhas.append({"modelo": nome, "AUC one-hot": onehot["auc"], "AUC ordinal": auc_ord,
                   "delta": round(auc_ord - onehot["auc"], 4),
                   "ep bootstrap": next(r["ep"] for r in SM["bootstrap"]
                                        if r["modelo"] == nome),
                   "nós one-hot": onehot["n_parametros"],
                   "nós ordinal": S.n_params(nome, modelo)})
print()
print(SP.tabela(pd.DataFrame(linhas), 4))

matriz ordinal: 40 colunas (one-hot: 124)
códigos -1 (nível ausente) entre as categóricas: 0



  modelo AUC one-hot AUC ordinal   delta ep bootstrap nós one-hot nós ordinal
  arvore      0,7041      0,7115  0,0074       0,0040       1.507       1.523
floresta      0,7403      0,7386 -0,0017       0,0039   2.736.410   3.262.836


Dois resultados desconfortáveis, e é para isso que a seção existe:

- a **árvore ordinal fica acima da one-hot**, por mais de um erro-padrão
  bootstrap (a coluna está ao lado do delta). O one-hot custa AUC aqui, e
  o card não pode dizer que ele é gratuito;
- a floresta quase não se move e paga em nós — as duas últimas colunas.

A escolha do one-hot, então, não se sustenta na AUC: sustenta-se na frase
que a regra vira. `dispneia_sim <= 0.5` se lê em voz alta;
`cs_raca <= 2.5` é uma regra sobre a ORDEM em que os níveis foram
catalogados, e uma explicação construída sobre ela mente com a AUC do
lado dela. É uma escolha de legibilidade **com o preço medido** — que é
outra coisa que uma escolha de legibilidade com o preço presumido zero.

## §4 — O preço da determinância

`FIXOS` trava `subsample` e `colsample_bytree` em 1,0 — sem subamostra
estocástica. A razão vem dos módulos 01 e 02: uma explicação que muda
quando a semente muda não é explicação, e o ceteris paribus de um modelo
que reamostra a cada fit vira ruído com cara de efeito.

Isso é uma restrição, e restrição custa. A célula mede o custo em vez de
supor que ele é zero: o vencedor com `subsample=1,0` e com
`subsample=0,8`, cada um em três sementes, mais a correlação entre as
previsões das sementes dentro de cada braço.

In [7]:
SEMENTES = (42, 43, 44)
arms, preds = [], {}
for sub in (1.0, 0.8):
    for semente in SEMENTES:
        cfg = dict(S.MELHORES["xgb"]) | {"subsample": sub, "random_state": semente}
        modelo, segundos, auc = auc_val("xgb", cfg)
        preds[(sub, semente)] = S.predict("xgb", modelo, g, va)
        arms.append({"subsample": sub, "semente": semente, "auc_val": auc,
                     "fit (s)": segundos})
braços = pd.DataFrame(arms)
print(SP.tabela(braços, {"subsample": 1, "auc_val": 4, "fit (s)": 1}))

print("\ncorrelação das previsões entre sementes, dentro de cada braço:")
for sub in (1.0, 0.8):
    pares = [(a, b) for i, a in enumerate(SEMENTES) for b in SEMENTES[i + 1:]]
    cors = [float(np.corrcoef(preds[(sub, a)], preds[(sub, b)])[0, 1]) for a, b in pares]
    iguais = all(np.array_equal(preds[(sub, SEMENTES[0])], preds[(sub, s)])
                 for s in SEMENTES[1:])
    print(f"  subsample={SP.pt(sub, 1)}: min {SP.pt(min(cors), 4)}  max {SP.pt(max(cors), 4)}"
          f"   previsões idênticas bit a bit? {iguais}")

media = braços.groupby("subsample")["auc_val"].mean().round(4)
ep_xgb = next(r["ep"] for r in SM["bootstrap"] if r["modelo"] == "xgb")
print(f"\nAUC média: subsample 1,0 -> {SP.pt(media[1.0], 4)}   0,8 -> {SP.pt(media[0.8], 4)}")
print(f"o que a subamostra compraria (0,8 - 1,0): {SP.pt(media[0.8] - media[1.0], 4, sinal=True)}")
print(f"1 EP bootstrap da AUC de validação (registro): {SP.pt(ep_xgb, 4)}")
print(f"em erros-padrão: {SP.pt((media[0.8] - media[1.0]) / ep_xgb, 2)} EP")

subsample semente auc_val fit (s)
      1,0      42  0,7564     4,5
      1,0      43  0,7564     4,4
      1,0      44  0,7564     4,2
      0,8      42  0,7574     4,6
      0,8      43  0,7577     4,6
      0,8      44  0,7578     4,7

correlação das previsões entre sementes, dentro de cada braço:
  subsample=1,0: min 1,0000  max 1,0000   previsões idênticas bit a bit? True
  subsample=0,8: min 0,9920  max 0,9938   previsões idênticas bit a bit? False

AUC média: subsample 1,0 -> 0,7564   0,8 -> 0,7576
o que a subamostra compraria (0,8 - 1,0): +0,0012
1 EP bootstrap da AUC de validação (registro): 0,0039
em erros-padrão: 0,31 EP


O braço determinístico entrega previsões **idênticas bit a bit** entre
sementes — não parecidas, idênticas — enquanto o braço com subamostra se
move na terceira casa a cada semente. Esse é o preço que os módulos 01 e
02 não podem pagar: um ceteris paribus que muda de resposta com a semente
não é uma explicação, é uma amostra de uma distribuição de explicações.

E a subamostra **compra** AUC, não custa: a média dos três braços de 0,8
fica acima da de 1,0. A célula imprime esse ganho em unidades de
erro-padrão bootstrap, e ele é uma fração de um EP — pela regra 2 do
critério isso é empate técnico, e é a regra 5 (determinância) que
desempata. O `FIXOS` não está escondendo um custo: está aplicando um
desempate declarado, com o número na mesa.

## §5 — A maldição do vencedor

Trinta configurações competiram na mesma validação; a vencedora é, em
parte, a que teve mais sorte nas linhas de 2023. A pergunta honesta é
quanto dessa vantagem sobrevive a 2024 — e o registro não responde: o
placar só leu o teste para o vencedor, que é exatamente a disciplina do
item 6.

Então a resposta é medida aqui, e **rotulada**: as duas vice-campeãs são
reajustadas e o teste é lido UMA vez para cada. São leituras
exploratórias — aconteceram depois de a escolha estar registrada, não
podem mudá-la, e não entram em card nenhum.

In [8]:
top = pd.DataFrame(SM["buscas"]["xgb"])[["posto", "config", "auc"]].head(8)
top["config"] = top["config"].map(compacta)
print("AUC de validação por posto (registro):")
print(SP.tabela(top, {"auc": 4}))
ep_xgb = next(r["ep"] for r in SM["bootstrap"] if r["modelo"] == "xgb")
print(f"\nvantagem do posto 1 sobre o posto seguinte:"
      f" {SP.pt(top['auc'].iloc[0] - top['auc'].iloc[1], 4, sinal=True)}")
print(f"1 EP bootstrap da AUC de validação: {SP.pt(ep_xgb, 4)}")
print("três configurações empatam no posto 2 (rank method='min'), por isso não")
print("existe posto 3 nem 4 nesta tabela.")

AUC de validação por posto (registro):
posto                                                                              config    auc
    1 learning_rate=0.05 max_depth=4 min_child_weight=100 n_estimators=800 reg_lambda=5.0 0,7564
    2   learning_rate=0.05 max_depth=5 min_child_weight=1 n_estimators=400 reg_lambda=5.0 0,7545
    2 learning_rate=0.05 max_depth=8 min_child_weight=100 n_estimators=800 reg_lambda=1.0 0,7545
    2   learning_rate=0.08 max_depth=5 min_child_weight=1 n_estimators=400 reg_lambda=5.0 0,7545
    5   learning_rate=0.05 max_depth=3 min_child_weight=5 n_estimators=800 reg_lambda=1.0 0,7540
    6   learning_rate=0.03 max_depth=4 min_child_weight=5 n_estimators=800 reg_lambda=5.0 0,7534
    7   learning_rate=0.08 max_depth=3 min_child_weight=1 n_estimators=400 reg_lambda=5.0 0,7528
    8   learning_rate=0.15 max_depth=3 min_child_weight=5 n_estimators=400 reg_lambda=0.5 0,7527

vantagem do posto 1 sobre o posto seguinte: +0,0019
1 EP bootstrap da AUC de validação:

In [9]:
# LEITURA EXPLORATÓRIA ROTULADA (item 6 do critério): o teste é lido uma
# vez para cada vice-campeã, DEPOIS da escolha, e não pode mudá-la.
gaps = [{"posto": 1, "config": compacta(SM["buscas"]["xgb"][0]["config"]),
         "auc_val": PLACAR[("xgb", "val")]["auc"],
         "auc_test": PLACAR[("xgb", "test")]["auc"],
         "leitura": "a única do protocolo"}]
for row in SM["buscas"]["xgb"][1:3]:
    cfg = json.loads(row["config"])
    modelo, _, auc_v = auc_val("xgb", cfg)
    auc_t = S.score(y_te, S.predict("xgb", modelo, g, te))["auc"]
    gaps.append({"posto": row["posto"], "config": compacta(row["config"]),
                 "auc_val": auc_v, "auc_test": auc_t, "leitura": "exploratória"})
quadro = pd.DataFrame(gaps)
quadro["delta val->test"] = (quadro["auc_test"] - quadro["auc_val"]).round(4)
print(SP.tabela(quadro, {"auc_val": 4, "auc_test": 4, "delta val->test": 4}))

posto                                                                              config auc_val auc_test              leitura delta val->test
    1 learning_rate=0.05 max_depth=4 min_child_weight=100 n_estimators=800 reg_lambda=5.0  0,7564   0,7644 a única do protocolo          0,0080
    2   learning_rate=0.05 max_depth=5 min_child_weight=1 n_estimators=400 reg_lambda=5.0  0,7545   0,7622         exploratória          0,0077
    2 learning_rate=0.05 max_depth=8 min_child_weight=100 n_estimators=800 reg_lambda=1.0  0,7545   0,7622         exploratória          0,0077


A vantagem do posto 1 sobre o posto seguinte, impressa na célula
anterior, é MENOR que um erro-padrão bootstrap — a definição operacional
de "não dá para distinguir". Em 2024 as três sobem quase o mesmo tanto e
a ordem se mantém: a maldição do vencedor **não mordeu aqui**, e as duas
vice-campeãs, que empatam na validação, seguem empatadas no teste.

Isso é confirmação, não prova. O critério tem a regra 2 (empate técnico
dentro de 1 EP) exatamente para não premiar a quarta casa decimal: a
margem do vencedor sobre as vice-campeãs continua sendo ruído da
validação, e o que 2024 mostra é que o ruído não se inverteu desta vez —
não que a margem fosse real.

## §6 — O que o mercado faria: retreinar em treino+val

A prática default antes de um deploy é reajustar o modelo escolhido em
tudo que existe — treino mais validação — para não jogar fora um ano de
dados. Aqui isso destruiria o único conjunto de seleção honesto: os
módulos 01–05 explicam um modelo que nunca viu 2023, e uma explicação
sobre um modelo que viu tudo não teria contra o que ser conferida.

Quanto custa não retreinar? A célula mede: mesmo vencedor, mesmos
hiperparâmetros, treino em `train + val`, teste lido uma vez —
exploratória, de novo.

In [10]:
x = S.matrices(g)["xgb"]
tv = (g["split"] != "test").to_numpy()
modelo_tv = S.build("xgb", S.MELHORES["xgb"])
modelo_tv.fit(x[tv], y[tv])
auc_tv = S.score(y_te, modelo_tv.predict_proba(x[te])[:, 1])
oficial = PLACAR[("xgb", "test")]
obs = oficial["observado"]
print(SP.tabela(pd.DataFrame([
    {"treino": "só train (<=2022) — o modelo do curso", "n_treino": int(tr.sum()),
     "auc_test": oficial["auc"], "brier_test": oficial["brier"],
     "previsto_medio": oficial["previsto_medio"],
     "gap (previsto - observado)": round(oficial["previsto_medio"] - obs, 4)},
    {"treino": "train + val (<=2023) — leitura exploratória", "n_treino": int(tv.sum()),
     "auc_test": auc_tv["auc"], "brier_test": auc_tv["brier"],
     "previsto_medio": auc_tv["previsto_medio"],
     "gap (previsto - observado)": round(auc_tv["previsto_medio"] - obs, 4)},
]), 4))
print(f"\ndelta de AUC no teste: {SP.pt(auc_tv['auc'] - oficial['auc'], 4, sinal=True)}"
      f"   (observado no teste: {SP.pt(obs, 4)})")

                                     treino n_treino auc_test brier_test previsto_medio gap (previsto - observado)
      só train (<=2022) — o modelo do curso  200.000   0,7644     0,1308         0,2154                     0,0330
train + val (<=2023) — leitura exploratória  224.148   0,7700     0,1285         0,1804                    -0,0020

delta de AUC no teste: +0,0056   (observado no teste: 0,1824)


O ano a mais compra um fio de AUC — e a **calibração inteira**. A coluna
do gap quase zera: 2023 já está do lado de cá da deriva, e o modelo para
de achar que 2024 tem a letalidade de 2021. É o ganho mais bonito deste
caderno, e é justamente o que o curso recusa.

Recusa de olhos abertos, com o número medido em vez de suposto: um modelo
treinado até 2023 não tem mais nenhum conjunto onde a seleção possa ser
feita honestamente, e o §2 do walkthrough mostra o que acontece quando se
seleciona sem um. A alternativa não é "ganhar um pouco de AUC" — é perder
a única estimativa que diz se ganhou.

## §7 — A floresta de fábrica

`RandomForestClassifier()` sem tocar em nada, só semente e `n_jobs`: o
default que o mercado chama de baseline. Ele não precisa ser reajustado
aqui — a sondagem está no registro, e o que ela mostra não é uma derrota
de AUC.

In [11]:
fabrica = SM["floresta_default"]
tunada = PLACAR[("floresta", "val")]
print(SP.tabela(pd.DataFrame([
    {"floresta": "de fábrica (defaults)", "auc": fabrica["auc"], "ece": fabrica["ece"],
     "brier": fabrica["brier"], "nós": fabrica["n_parametros"], "fit_s": fabrica["fit_s"]},
    {"floresta": "tunada (MELHORES)", "auc": tunada["auc"], "ece": tunada["ece"],
     "brier": tunada["brier"], "nós": tunada["n_parametros"], "fit_s": tunada["fit_s"]},
]), {"auc": 4, "ece": 4, "brier": 4, "fit_s": 1}))
print(f"\nconfiguração de fábrica: {fabrica['config']}")
print(f"nós a mais que a tunada: {SP.pt_int(fabrica['n_parametros'] - tunada['n_parametros'])}")
print(f"parâmetros de MELHORES['floresta']: {json.dumps(S.MELHORES['floresta'], sort_keys=True)}")

             floresta    auc    ece  brier        nós fit_s
de fábrica (defaults) 0,7219 0,1004 0,1501 11.222.316   8,5
    tunada (MELHORES) 0,7403 0,0897 0,1454  2.736.410  18,1

configuração de fábrica: RandomForestClassifier() — só n_jobs e random_state
nós a mais que a tunada: 8.485.906
parâmetros de MELHORES['floresta']: {"max_depth": 16, "max_features": "sqrt", "min_samples_leaf": 5}


O default cresce cada árvore até a folha pura: milhões de nós sobre 200
mil linhas. O preço aparece primeiro no custo e na **calibração** — a
coluna `ece` — e só depois na AUC, que é a coluna que quase todo mundo
olha. Um baseline "sem tunar" que sai com esse ECE não é um baseline
neutro: é uma escolha de hiperparâmetros feita pelo autor da biblioteca,
com consequências que o placar do curso precisa enxergar.

## §8 — O vencedor sobrevive à base cheia?

O estudo inteiro rodou na amostra commitada (treino subamostrado). A
pergunta que autoriza o desenho do curso é se o vencedor continua
vencedor com 1,24 M de linhas de treino — se a resposta fosse não, o
modelo que os módulos explicam seria um artefato da amostra.

Val e teste são os mesmos nos dois casos (a amostra os carrega inteiros),
então o único que muda é o treino. Nada de `reindex` é preciso aqui: o
XGBoost recebe as categóricas nativas e o modelo é ajustado e avaliado
sobre a MESMA moldura, com os mesmos níveis de categoria.

In [12]:
if ouro is None:
    print("PULADO: exige o Ouro completo em ~/Documents/srag-data/gold/.")
    print("Os outputs commitados vêm de uma execução local com a base cheia.")
else:
    x_cheio, _, y_cheio = M.design_matrix(ouro)
    tr_cheio = (ouro["split"] == "train").to_numpy()
    modelo_cheio = S.build("xgb", S.MELHORES["xgb"])
    modelo_cheio.fit(x_cheio[tr_cheio], y_cheio[tr_cheio])

    linhas = []
    for split in ("val", "test"):
        m = (ouro["split"] == split).to_numpy()
        vivo = S.score(y_cheio[m], modelo_cheio.predict_proba(x_cheio[m])[:, 1])
        amostra = PLACAR[("xgb", split)]
        linhas.append({"split": split, "n": int(m.sum()),
                       "auc amostra": amostra["auc"], "auc cheio": vivo["auc"],
                       "delta auc": round(vivo["auc"] - amostra["auc"], 4),
                       "ece amostra": amostra["ece"], "ece cheio": vivo["ece"],
                       "previsto amostra": amostra["previsto_medio"],
                       "previsto cheio": vivo["previsto_medio"]})
    print(f"treino cheio: {SP.pt_int(int(tr_cheio.sum()))} linhas"
          f" (amostra: {SP.pt_int(int(tr.sum()))})")
    print(SP.tabela(pd.DataFrame(linhas), 4))

treino cheio: 1.242.680 linhas (amostra: 200.000)
split      n auc amostra auc cheio delta auc ece amostra ece cheio previsto amostra previsto cheio
  val 24.148      0,7564    0,7601    0,0037      0,0316    0,0360           0,2209         0,2253
 test 16.142      0,7644    0,7680    0,0036      0,0329    0,0372           0,2154         0,2196


O delta de AUC é pequeno e do mesmo sinal nos dois splits: seis vezes
mais treino compra a terceira casa, não um modelo diferente. É esse
número que autoriza a frase do desenho do curso — explicar o
modelo-da-amostra é explicar o modelo — e ele fica medido aqui em vez de
suposto.

A calibração vai para o outro lado, e vale reparar: o `ece` do modelo
cheio é PIOR nos dois splits. Faz sentido — mais treino aqui é mais
2020–22 dentro do modelo, e o excesso de previsão é justamente a herança
daquele regime. A amostra não está escondendo nada do curso; se algo, ela
suaviza um pouco a deriva que os módulos vão explicar. A leitura do teste
nesta seção é exploratória, como as do §5.

## §9 — O que mudou no modelo do curso

A regra 8 do critério dizia, antes de qualquer número: se o vencedor
tunado divergir de `XGB_PARAMS`, o modelo do curso muda. Ele divergiu, e
mudou. A célula abaixo imprime os dois lados e o que a troca arrastou
junto — o paciente-exemplo, que é escolhido por uma regra aplicada ao
modelo, e portanto muda quando o modelo muda.

In [13]:
mm = json.loads(
    (REPO / "modules/00-dataset/gold/model_metrics.json").read_text(encoding="utf-8")
)
antes = SM["consequencia"]["xgb_params_hoje"]
depois = SM["consequencia"]["xgb_params_vencedor"]
print(SP.tabela(pd.DataFrame([{"parâmetro": k,
                               "antes do estudo": repr(antes[k]),
                               "vencedor (adotado)": repr(depois[k]),
                               "em XGB_PARAMS hoje": repr(M.XGB_PARAMS[k])}
                              for k in sorted(antes)])))
assert {k: M.XGB_PARAMS[k] for k in depois} == depois, "a adoção não está em XGB_PARAMS"

print(f"\nexemplar do curso sob os parâmetros adotados:"
      f" gold_id {mm['exemplar']['gold_id']}, p = {SP.pt(mm['exemplar']['p_obito'], 3)}")
print(f"  {mm['exemplar']['resumo']}")
print(f"paciente 2021: gold_id {mm['paciente_2021']['gold_id']},"
      f" p = {SP.pt(mm['paciente_2021']['p_obito'], 3)}")
print("\npendência registrada: os cinco módulos de método já escritos (01-05,")
print("três deles entraram na main enquanto este estudo rodava) foram")
print("construídos sobre o modelo anterior e citam números dele em prosa; a")
print("re-sincronização é PR posterior, com re-execução e conferência número a")
print("número — a consequência estava declarada antes de medir.")

    parâmetro antes do estudo vencedor (adotado) em XGB_PARAMS hoje
learning_rate            0.08               0.05               0.05
    max_depth               5                  4                  4
 n_estimators             400                800                800

exemplar do curso sob os parâmetros adotados: gold_id 1276776, p = 0,500
  90 anos, masculino, SE, início em 2024 (semana 34), 1 dose(s) antes do sintoma, comorbidades: cardiopati
paciente 2021: gold_id 834192, p = 0,500

pendência registrada: os cinco módulos de método já escritos (01-05,
três deles entraram na main enquanto este estudo rodava) foram
construídos sobre o modelo anterior e citam números dele em prosa; a
re-sincronização é PR posterior, com re-execução e conferência número a
número — a consequência estava declarada antes de medir.


O que este caderno deixa registrado, e que o card não carrega: a busca
reproduz (§2), o one-hot custa AUC na árvore (§3), a determinância custa
uma fração de um erro-padrão (§4), a margem sobre as vice-campeãs é da
ordem do erro-padrão e sobrevive a 2024 (§5), retreinar em treino+val
compraria sobretudo calibração (§6), o default da floresta paga em
calibração antes de pagar em AUC (§7) e o vencedor sobrevive à base cheia
com a terceira casa (§8).

Nenhum desses números muda a escolha — ela está registrada e fechada. Eles
existem para que a próxima pessoa que quiser mudá-la saiba o que vai ter
de medir de novo.